### Importation des données

In [1]:
import sys
sys.path.append('../src')
import utils
import pandas as pd
import s3fs
import geopandas as gpd

Notre première base de données correspond aux arbres dans la ville de Paris. 

In [2]:
arbres = pd.read_csv("https://www.data.gouv.fr/api/1/datasets/r/60433484-f30e-44ef-a362-e5553a9b7a42", sep = ";")


In [3]:
arbres

,IDBASE,TYPE EMPLACEMENT,DOMANIALITE,ARRONDISSEMENT,COMPLEMENT ADRESSE,LIEU / ADRESSE,IDEMPLACEMENT,LIBELLE FRANCAIS,GENRE,ESPECE,VARIETE OUCULTIVAR,CIRCONFERENCE (cm),HAUTEUR (m),STADE DE DEVELOPPEMENT,REMARQUABLE,geo_point_2d
0,267615,Arbre,Alignement,PARIS 1ER ARRDT,4 f,QUAI FRANCOIS MITTERRAND,000101033,Platane,Platanus,x hispanica,NaN,322,20,Adulte,NON,"48.85936815146502, 2.336197260919396"
1,2001356,Arbre,Alignement,BOIS DE BOULOGNE,Candélabre n°7281,ALLEE DE LA REINE MARGUERITE,001202097,Marronnier,Aesculus,hippocastanum,NaN,161,15,Adulte,NON,"48.87256400936489, 2.254016470558061"
2,2015583,Arbre,Jardin,BOIS DE VINCENNES,NaN,PARC FLORAL DE PARIS / ROUTE DE LA PYRAMIDE,02910005,Magnolia,Magnolia,grandiflora,NaN,80,8,Adulte,NON,"48.837160227435746, 2.4439083626402907"
3,2021315,Arbre,DASCO,PARIS 20E ARRDT,NaN,ECOLE MATERNELLE / 20 RUE DES CENDRIERS,063902002,Poirier à fruits,Pyrus,communis,''Doyenné du Comice'' COMICE,20,2,Jeune (arbre),NON,"48.86520599592482, 2.3867746391040847"
4,2054405,Arbre,Alignement,PARIS 13E ARRDT,NaN,PLACE NATIONALE,000304001,Tilleul,Tilia,x europaea,NaN,35,5,Jeune (arbre),NON,"48.82951463924067, 2.365357007521199"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
217747,283255,Arbre,Alignement,PARIS 19E ARRDT,NaN,PLACE DE JOINVILLE,000101006,Platane,Platanus,x hispanica,NaN,55,7,Jeune (arbre)Adulte,NON,"48.88963327830016, 2.380303666095808"
217748,2024286,Arbre,Jardin,BOIS DE VINCENNES,L-08,ARBORETUM DE PARIS / 50 ROUTE DE LA PYRAMIDE,05-719480719,Cèdre,Cedrus,atlantica,''Atlantica Pendula'',0,0,NaN,NON,"48.82132687128179, 2.457343421639024"
217749,261246,Arbre,Alignement,PARIS 17E ARRDT,NaN,RUE CURNONSKY,000202001,Erable,Acer,platanoides,''Globosum'',33,3,Jeune (arbre),NON,"48.89093410862582, 2.2971646942176833"
217750,232032,Arbre,Alignement,PARIS 19E ARRDT,NaN,RUE DE BELLEVILLE,000103011,Sophora,Styphnolobium,japonicum,NaN,120,10,Adulte,NON,"48.87537993953184, 2.394283352566337"


Notre deuxième base de données correspond aux iris de Paris, triés depuis les iris de la France métropolitaine. 

In [ ]:
fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})

MY_BUCKET = "raphcrre"
PATH_IRIS = f"{MY_BUCKET}/diffusion/projet_arbres/iris.gpkg"

with fs.open(PATH_IRIS, 'rb') as f:
    iris_france = gpd.read_file(f)

iris = iris_france[iris_france['code_insee'].astype(str).str.startswith('75')].copy()

In [ ]:
iris

### Jointure

Réalisons une jointure entre nos tables pour avoir les informations sur les iris de chaque arbre. 

In [ ]:
df_arbres = utils.jointure_arbres_iris(arbres, iris)


### Nettoyage de la base de données

On remarque que notre table contient trop de colonnes inutiles : 

- IDBASE : id de la ligne inutile
- cleabs : Identifiant technique IGN
- IDEMPLACEMENT, COMPLEMENT ADRESSE, LIEU / ADRESSE : indications géographique inutiles comme on possède la localisation. 
- iris : déjà contenu dans code_iris, qui est un indentifiant complet et unique. 
- geo_point_2d, lat et lon : redondant avec geometry
- 'ARRONDISSEMENT : variable doublée
- index_right : rajoutée avec la jointure
- TYPE EMPLACEMENT : modalité Arbre partout. 

In [ ]:
df_arbres = utils.suppression_colonnes(df_arbres)
df_arbres

In [ ]:
print(df_arbres.columns.tolist())

Suppression des valeurs manquantes et traitement des valeurs aberrantes de circonférence et de hauteur d'arbre. 

In [ ]:
df_arbres = utils.nettoyer_valeurs_aberrantes(df_arbres)

print("-" * 30)
print(f"Statistiques après nettoyage :")
print(f"Hauteur moyenne : {df_arbres['hauteur_m'].mean():.2f} m")
print(f"Circonférence moyenne : {df_arbres['circonference_cm'].mean():.2f} cm")
print("-" * 30)